In [10]:
import torch
from torchvision import models, transforms
from PIL import Image
import cv2
import numpy as np
import urllib.request
import time
import os
import glob

In [11]:
if not os.path.exists("./my_test_images"):
    raise FileNotFoundError(
        "\nPlease make sure you have the required input in this directory."
    )

if not os.path.exists("imagenet_classes.txt"):
    print("Downloading class labels...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
        "imagenet_classes.txt"
    )

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]



In [12]:
#MODEL SETUP
print("Loading pre-trained MobileNetV2 model...")

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.eval()
#Using mobilenet insted of resnet18.
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0)

    start_time = time.perf_counter()
    with torch.no_grad():
        output = model(input_tensor)
    end_time = time.perf_counter()
    latency_ms = (end_time - start_time) * 1000.0

    probs = torch.nn.functional.softmax(output[0], dim=0)
    confidence, predicted_idx = torch.max(probs, 0)

    return categories[predicted_idx.item()], confidence.item(), latency_ms



Loading pre-trained MobileNetV2 model...


Implement the following functions:

1.   List item
2.   List item



In [13]:
def simulate_turbidity(img_array):
    """
    Simulate murky water (blur/haze).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    kernel_size = (31, 31)

    # Applying Gaussian blur
    img_array= cv2.GaussianBlur(img_array, kernel_size, 0)
    return img_array


In [14]:
import cv2
import numpy as np

def simulate_color_shift(img_array):
    """
    Simulate depth color loss (attenuate the red channel).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    red_channel = img_array[:, :, 2].astype(np.float32)

    # I am assuming 80% of red light is absorbed
    red_channel = red_channel * 0.2
    img_array[:, :, 2] = np.clip(red_channel, 0, 255).astype(np.uint8)

    return img_array

In [15]:
def simulate_sensor_noise(img_array):
    """
    Simulate low-light digital camera noise.
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    noise = np.random.normal(0, 50, img_array.shape)
    # since there is high noise in murky water ,so taking noise as 50.

    img_array= img_array.astype(np.float32) + noise
    img_array = np.clip(img_array, 0, 255)
    img_array = img_array.astype(np.uint8)
    return img_array

In [16]:
if __name__ == "__main__":

    image_directory = "./my_test_images"
    output_directory = "output_images"

    """
    making an input file and an output file.
    So I need not to change test images mannualy and making code longer.
    """

    os.makedirs(output_directory, exist_ok=True)
    print(f"Look for your images exactly here: {os.path.abspath(output_directory)}")

    image_paths = glob.glob(os.path.join(image_directory, "*.png"))
    image_paths.extend(glob.glob(os.path.join(image_directory, "*.jpg")))

    print("\nRUNNING VISION DIAGNOSTICS FOR MULTIPLE IMAGES")
    print("=" * 50)

    for base_image in image_paths:
        print(f"\nEvaluating Image: {os.path.basename(base_image)}")

        img = cv2.imread(base_image)

        base_name = os.path.basename(base_image).split('.')[0]

        turbid_path = os.path.join(output_directory, f"test_turbid_{base_name}.jpg")
        colorshift_path = os.path.join(output_directory, f"test_colorshift_{base_name}.jpg")
        noise_path = os.path.join(output_directory, f"test_noise_{base_name}.jpg")

        cv2.imwrite(turbid_path, simulate_turbidity(img.copy()))
        cv2.imwrite(colorshift_path, simulate_color_shift(img.copy()))
        cv2.imwrite(noise_path, simulate_sensor_noise(img.copy()))

        images_to_test = [
            ("Baseline (Clean)", base_image),
            ("Turbidity", turbid_path),
            ("Color Shift", colorshift_path),
            ("Sensor Noise", noise_path)
        ]
        for condition_name, file_path in images_to_test:
            label, conf, latency = predict(file_path)
            print(f"Condition : {condition_name}")
            print(f"Prediction: {label}")
            print(f"Confidence: {conf:.4f}")
            print(f"Latency   : {latency:.2f} ms")
            print("-" * 30)

    print("\nALL DIAGNOSTICS COMPLETE")

Look for your images exactly here: C:\Users\airfo\PycharmProjects\first\output_images

RUNNING VISION DIAGNOSTICS FOR MULTIPLE IMAGES

Evaluating Image: set_f20_SESR.png
Condition : Baseline (Clean)
Prediction: coral fungus
Confidence: 0.0452
Latency   : 8.87 ms
------------------------------
Condition : Turbidity
Prediction: European fire salamander
Confidence: 0.0536
Latency   : 8.27 ms
------------------------------
Condition : Color Shift
Prediction: rock beauty
Confidence: 0.3153
Latency   : 8.05 ms
------------------------------
Condition : Sensor Noise
Prediction: coral reef
Confidence: 0.0506
Latency   : 12.31 ms
------------------------------

Evaluating Image: set_f46_SESR.png
Condition : Baseline (Clean)
Prediction: king crab
Confidence: 0.1118
Latency   : 7.76 ms
------------------------------
Condition : Turbidity
Prediction: tarantula
Confidence: 0.0656
Latency   : 8.20 ms
------------------------------
Condition : Color Shift
Prediction: sea snake
Confidence: 0.1660
Late